In [1]:
import warnings
warnings.filterwarnings("ignore")

In [3]:
!pip install chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 67.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 84.9 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 72.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.9/178.9 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 3.2 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-proto
    Found existing installation: opentelemetry-proto 1.38.

In [4]:
import os
import numpy as np
import pandas as pd

import chromadb
# from sentence_transformers import SentenceTransformer
from tqdm import tqdm

In [6]:
import shutil

src = "/kaggle/input/datasets/doubleogon/housing-vector-db/housing_vector_db"
dst = "/kaggle/working/housing_vector_db"

shutil.copytree(src, dst)

'/kaggle/working/housing_vector_db'

In [7]:
chroma_client = chromadb.PersistentClient(path="/kaggle/working/housing_vector_db")

collection = chroma_client.get_collection(
    name="property_recommendations"
)

In [8]:
print(collection.count())

10000


In [12]:
src = "/kaggle/input/datasets/doubleogon/eval-dataset/eval_dataset.csv"
dst = "/kaggle/working/eval_dataset.csv"

shutil.copy(src, dst)

'/kaggle/working/eval_dataset.csv'

In [16]:
df_eval = pd.read_csv("/kaggle/working/eval_dataset.csv")

In [17]:
def chromadb_recommend(query_text, max_price=5000, min_beds=0, top_n=5):
    """
    Performs true deep semantic search combined with structural database constraints.
    """
    # Run the query directly against the vector database
    results = collection.query(
        query_texts=[query_text],
        n_results=top_n,
        where={
            "$and": [
                {"price": {"$lte": max_price}},
                {"bedrooms": {"$gte": min_beds}}
            ]
        }
    )
    
    # Re-structure the output back into a pretty Pandas DataFrame for display
    recommendations = []
    if not results or not results['ids'] or len(results['ids'][0]) == 0:
        return pd.DataFrame() # Return empty DF if no matches found
        
    for i in range(len(results['ids'][0])):
        rec = {
            "id": results['ids'][0][i],
            "distance_score": results['distances'][0][i], 
            **results['metadatas'][0][i]
        }
        recommendations.append(rec)
        
    return pd.DataFrame(recommendations)

In [18]:
def get_hit_rate(retrieved_ids, expected_ids):
    # Convert all IDs to string and strip spaces to prevent type mismatch (e.g., 101 vs "101")
    clean_retrieved = [str(x).strip() for x in retrieved_ids]
    clean_expected = [str(x).strip() for x in expected_ids]
    
    has_overlap = set(clean_retrieved).intersection(set(clean_expected))
    return 1 if len(has_overlap) > 0 else 0

def get_mrr(retrieved_ids, expected_ids):
    clean_retrieved = [str(x).strip() for x in retrieved_ids]
    clean_expected = [str(x).strip() for x in expected_ids]
    
    for rank, house_id in enumerate(clean_retrieved, start=1):
        if house_id in clean_expected:
            return 1.0 / rank
    return 0.0

In [22]:
import time

evaluation_results = []

for idx, row in tqdm(df_eval.iterrows(), total=len(df_eval), desc="Evaluating Engine"):
    start_time = time.time()
    
    df_recommendations = chromadb_recommend(
        query_text=row["query_text"],
        max_price=row["max_price"],
        min_beds=row["min_beds"],
        top_n=5
    )
    
    latency = (time.time() - start_time) * 1000 # in milliseconds
    
    # Extract IDs retrieved by the system
    retrieved_ids = df_recommendations["id"].tolist() if not df_recommendations.empty else []
    
    # Calculate quality scores
    hit_rate = get_hit_rate(retrieved_ids, [row["expected_ids"]])
    mrr = get_mrr(retrieved_ids, [row["expected_ids"]]
                 )
    
    evaluation_results.append({
        "query": row["query_text"],
        "retrieved": retrieved_ids,
        "expected": row["expected_ids"],
        "hit_rate": hit_rate,
        "mrr": mrr,
        "latency_ms": latency
    })

df_metrics = pd.DataFrame(evaluation_results)

Evaluating Engine: 100%|██████████| 30/30 [00:02<00:00, 14.69it/s]


In [24]:
print("\n================ METRICS REPORT ================")
print(f"Overall Hit Rate @ 5 : {df_metrics['hit_rate'].mean() * 100:.1f}%")
print(f"Mean Reciprocal Rank : {df_metrics['mrr'].mean():.3f}")
print(f"Average Latency      : {df_metrics['latency_ms'].mean():.2f} ms")
print("================================================\n")

df_metrics[["query", "retrieved", "expected", "hit_rate", "mrr", "latency_ms"]]


================ METRICS REPORT ================
Overall Hit Rate @ 5 : 93.3%
Mean Reciprocal Rank : 0.900
Average Latency      : 67.19 ms



,query,retrieved,expected,hit_rate,mrr,latency_ms
0,What apartment is located at 2nd St NE in Wash...,"[5668626895, 5668624951, 5668632832, 566863360...",5668626895,1,1.0,55.397272
1,What's the monthly rental rate for this unit a...,"[5664597177, 5668632677, 5668618648, 566863902...",5664597177,1,1.0,55.318832
2,"N Scott St, 14th St N, Arlington, VA 22209, Ar...","[5668626833, 5668626759, 5664574816, 566862709...",5668626833,1,1.0,92.681408
3,What's the monthly rental rate for this apartm...,"[5659918074, 5668627023, 5668621056, 566457663...",5659918074,1,1.0,58.950424
4,"Washington Blvd, N Cleveland St, Arlington, Ar...","[5668626759, 5664574816, 5668626687, 566457132...",5668626759,1,1.0,55.703163
5,"RARE GEM with private outdoor terrace, availab...","[5509108930, 5667891676, 5508986256, 550911396...",5667891676,1,0.5,102.232695
6,What's the monthly rental rate for this unit l...,"[5668627426, 5668634809, 5668622824, 566459254...",5668627426,1,1.0,100.776434
7,"What apartment is located at Oak St NW, 16th S...","[5668626687, 5664575334, 5668623923, 566862495...",5668626687,1,1.0,53.112984
8,What is the rental rate for this unit located ...,"[5668610290, 5668633216, 5659915570, 566862398...",5668610290,1,1.0,88.527679
9,What's the rent in this apartment?,"[5509248658, 5508779254, 5509044554, 550902251...",5668627023,0,0.0,53.554773


## NDCG

In [25]:
from sklearn.metrics import ndcg_score

In [33]:
# Evaluation Loop
ndcg_scores = []
top_n = 5

for item in df_eval.to_dict("records"):
    query = item['query_text']
    max_price = item['max_price']
    min_beds = item['min_beds']
    expected_id = item['expected_ids']
    
    # Query chromadb_recommend function
    df_recommendations = chromadb_recommend(
        query_text=query, 
        max_price=max_price, 
        min_beds=min_beds, 
        top_n=top_n
    )
    
    # If database returns nothing or the expected ID is not found, NDCG is 0
    if df_recommendations.empty:
        ndcg_scores.append(0.0)
        continue
        
    expected_id = str(item["expected_ids"]).strip()

    retrieved_ids = [
    str(x).strip()
    for x in df_recommendations["id"].tolist()
    ]
    
    if expected_id not in retrieved_ids:
        ndcg_scores.append(0.0)
        continue
        
    # Create Binary Ground-Truth relevance array: 1 for expected ID, 0 for others
    # e.g., if expected ID is at index 2, relevance will be [0, 0, 1, 0, 0]
    true_relevance = [1 if rid == expected_id else 0 for rid in retrieved_ids]
    
    # Create predicted scoring descending list (tells sklearn that items are already ranked)
    # e.g., if we retrieved 5 items: [5, 4, 3, 2, 1]
    predicted_scores = list(range(len(retrieved_ids), 0, -1))
    
    # Calculate NDCG@5 using sklearn (expects 2D array input)
    score = ndcg_score(
        np.asarray([true_relevance]), 
        np.asarray([predicted_scores]), 
        k=top_n
    )
    ndcg_scores.append(score)

In [34]:
# Calculate and Print average NDCG@5
mean_ndcg = np.mean(ndcg_scores)
print("--- sklearn NDCG Evaluation ---")
print(f"Total Queries Evaluated: {len(df_eval)}")
print(f"Mean NDCG@{top_n}: {mean_ndcg:.4f}")

--- sklearn NDCG Evaluation ---
Total Queries Evaluated: 30
Mean NDCG@5: 0.9087


## Error Analysis

In [43]:
error_cases = []

top_n = 5

for item in df_eval.to_dict("records"):

    query = item["query_text"]
    max_price = item["max_price"]
    min_beds = item["min_beds"]
    expected_id = str(item["expected_ids"]).strip()

    results = chromadb_recommend(
        query_text=query,
        max_price=max_price,
        min_beds=min_beds,
        top_n=top_n
    )

    if results.empty:
        error_cases.append({
            "query_text": query,
            "expected_id": expected_id,
            "retrieved_ids": [],
            "rank": None,
            "failure_type": "No results returned"
        })
        continue


    retrieved_ids = [
        str(x).strip()
        for x in results["id"].tolist()
    ]


    # Check if expected result exists
    if expected_id not in retrieved_ids:

        error_cases.append({
            "query_text": query,
            "expected_id": expected_id,
            "retrieved_ids": retrieved_ids,
            "rank": None,
            "failure_type": "Expected item not retrieved"
        })

    else:
        rank = retrieved_ids.index(expected_id) + 1

        # Only save bad ranking cases
        if rank > 1:
            error_cases.append({
                "query_text": query,
                "expected_id": expected_id,
                "retrieved_ids": retrieved_ids,
                "rank": rank,
                "failure_type": "Relevant item ranked low"
            })


# Convert to dataframe
error_df = pd.DataFrame(error_cases)


# Save report
error_df.to_csv(
    "error_analysis.csv",
    index=False
)

In [44]:
print(f"Total error cases: {len(error_df)}")
print(error_df["failure_type"].value_counts())

Total error cases: 4
failure_type
Relevant item ranked low       2
Expected item not retrieved    2
Name: count, dtype: int64


In [36]:
error_df = pd.read_csv("/kaggle/working/error_analysis.csv")

In [42]:
display(error_df.style)

,query_text,expected_id,retrieved_ids,rank,failure_type
0,"RARE GEM with private outdoor terrace, available immediately, net effective with 2475 rental cost",5667891676,"['5509108930', '5667891676', '5508986256', '5509113967', '5668633859']",2.000000,Relevant item ranked low
1,What's the rent in this apartment?,5668627023,"['5509248658', '5508779254', '5509044554', '5509022513', '5668616896']",nan,Expected item not retrieved
2,"What's the rental rate for this apartment in Tucson, AZ?",5664598162,"['5668621906', '5668616744', '5668616688', '5668616721', '5668616710']",nan,Expected item not retrieved
3,What amenities does New Bern Studio include?,5659276240,"['5654898031', '5659276240', '5509036938', '5509030517', '5509092854']",2.000000,Relevant item ranked low


The retrieval system achieved 93.3% Hit Rate@5.  

Analysis of failed cases showed two main limitations:  
1) ambiguous queries lacking property context, and  
2) insufficient use of structured metadata such as location. 

Future improvements include hybrid keyword-semantic ranking and metadata-aware retrieval or metadata filter.

ref: https://www.evidentlyai.com/ranking-metrics/evaluating-recommender-systems#ranking-quality-metrics  

https://shiftasia.com/community/evaluating-vector-search-quality/